# Steepest Descent Method & Conditioning Analysis for SPD Linear Systems

This notebook implements the **Steepest Descent** iterative solver for solving symmetric positive definite (SPD) linear systems $Ax = b$, evaluates baseline convergence, computes the spectral condition number $\kappa_2(A)$, and tests sensitivity to right-hand side perturbations along extreme eigenvectors ($q_{\max}$ and $q_{\min}$).

## 1. Imports & Steepest Descent Solver Definition

For a symmetric positive definite (SPD) matrix $A \in \mathbb{R}^{n \times n}$, solving $Ax = b$ is equivalent to minimizing the quadratic energy functional:
$$\phi(x) = \frac{1}{2} x^T A x - b^T x$$

At each step $k$, the negative gradient equals the residual $r_k = b - A x_k$, and the optimal exact line-search step size along $r_k$ is:
$$\alpha_k = \frac{r_k^T r_k}{r_k^T A r_k}$$

The solver terminates once the relative residual satisfies $\frac{\|r_k\|_2}{\|b\|_2} \le \text{tol}$.

In [ ]:
import numpy as np


def steepest_descent(A, b, x0, tol, maxiter=200000):
    """Solve Ax = b for SPD matrix A using exact-line-search Steepest Descent."""
    x = np.asarray(x0, dtype=float).copy()
    r = b - A @ x
    bnorm = np.linalg.norm(b)
    history = [np.linalg.norm(r) / bnorm]

    for k in range(maxiter):
        if history[-1] <= tol:
            return x, np.array(history)

        Ar = A @ r
        denom = r @ Ar
        if denom <= 0:
            raise ValueError('A is not SPD')

        alpha = (r @ r) / denom
        x += alpha * r
        r -= alpha * Ar
        history.append(np.linalg.norm(r) / bnorm)

    raise RuntimeError('maximum iteration count reached')

## 2. Load Matrix & Baseline Solve ($b = A \cdot \mathbf{1}$)

We load $A$ from `A.dat`, verify numerical symmetry ($A = A^T$), and construct a synthetic right-hand side $b = A\mathbf{1}$ whose exact analytical solution is $x_{\text{exact}} = \mathbf{1}$. Starting from $x_0 = \mathbf{0}$ with tolerance $10^{-10}$, we measure iteration count, final relative residual, and relative forward solution error $\frac{\|x - \mathbf{1}\|_2}{\|\mathbf{1}\|_2}$.

In [ ]:
A = np.loadtxt('A.dat', delimiter=',')
assert np.allclose(A, A.T)

n = A.shape[0]
ones = np.ones(n)
zero = np.zeros(n)

b = A @ ones
x, residuals = steepest_descent(A, b, zero, 1e-10)
solution_error = np.linalg.norm(x - ones) / np.linalg.norm(ones)

print(f"Iterations: {len(residuals) - 1}")
print(f"Final relative residual: {residuals[-1]:.4e}")
print(f"Relative solution error: {solution_error:.4e}")

## 3. Eigenvalue Decomposition & Spectral Condition Number ($\kappa_2$)

Because $A$ is real symmetric, `np.linalg.eigh` computes its real eigenvalues $\lambda_1 \le \lambda_2 \le \dots \le \lambda_n$ and orthonormal eigenvectors $Q = [q_1, \dots, q_n]$. For an SPD matrix, the 2-norm condition number is:
$$\kappa_2(A) = \frac{\lambda_{\max}}{\lambda_{\min}}$$

Large values of $\kappa_2(A)$ cause ill-conditioned error amplification and slow zig-zagging convergence in steepest descent, governed by the bound $\frac{\kappa_2(A) - 1}{\kappa_2(A) + 1}$.

In [ ]:
eigenvalues, Q = np.linalg.eigh(A)
qmin, qmax = Q[:, 0], Q[:, -1]
kappa = eigenvalues[-1] / eigenvalues[0]

print(f"Min eigenvalue: {eigenvalues[0]:.4e}")
print(f"Max eigenvalue: {eigenvalues[-1]:.4e}")
print(f"Condition number (kappa): {kappa:.4e}")

## 4. Perturbation Test Along $q_{\max}$ (Best-Case Sensitivity)

Set the true solution to $x^* = q_{\max}$ so that $b^* = A x^* = \lambda_{\max} q_{\max}$ and $\|b^*\|_2 = \lambda_{\max}$. Perturbing $b^*$ by $\delta b = \eta \|b^*\|_2 q_{\max}$ yields:
$$\delta x = A^{-1} \delta b = \eta \frac{\lambda_{\max}}{\lambda_{\max}} q_{\max} = \eta q_{\max}$$

Thus, the relative error amplification factor $\frac{\|\tilde{x} - x^*\|_2 / \|x^*\|_2}{\|\delta b\|_2 / \|b^*\|_2}$ is approximately $1$ (minimal amplification).

In [ ]:
eta = 1e-4
xstar = qmax
bstar = A @ xstar

db_max = eta * np.linalg.norm(bstar) * qmax
xtilde_max, hist_max = steepest_descent(A, bstar + db_max, zero, 1e-12)

rel_b_max = np.linalg.norm(db_max) / np.linalg.norm(bstar)
rel_x_max = np.linalg.norm(xtilde_max - xstar) / np.linalg.norm(xstar)

print(rel_b_max, rel_x_max, rel_x_max / rel_b_max, hist_max[-1], len(hist_max) - 1)

## 5. Perturbation Test Along $q_{\min}$ (Worst-Case Sensitivity)

Keeping $x^* = q_{\max}$ ($\|b^*\|_2 = \lambda_{\max}$) and perturbing $b^*$ in the direction of the smallest eigenvector $\delta b = \eta \|b^*\|_2 q_{\min}$ produces:
$$\delta x = A^{-1} \delta b = \eta \frac{\lambda_{\max}}{\lambda_{\min}} q_{\min} = \eta \kappa_2(A) q_{\min}$$

Here, the relative solution error is amplified by the full condition number $\kappa_2(A)$, achieving the theoretical worst-case perturbation bound.

In [ ]:
db_min = eta * np.linalg.norm(bstar) * qmin
xtilde_min, hist_min = steepest_descent(A, bstar + db_min, zero, 1e-12)

rel_b_min = np.linalg.norm(db_min) / np.linalg.norm(bstar)
rel_x_min = np.linalg.norm(xtilde_min - xstar) / np.linalg.norm(xstar)

print(rel_b_min, rel_x_min, rel_x_min / rel_b_min, hist_min[-1], len(hist_min) - 1)